In [1]:
import os

REPO_URL = "https://github.com/HICAI-ZJU/KANO.git"
REPO_DIR = "./KANO"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("[OK] KANO repo already exists:", os.path.abspath(REPO_DIR))


Cloning into './KANO'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done. Counting objects:  87% (131/150)
remote: Compressing objects: 100% (131/131), done.Compressing objects:  40% (53/131)remote: Compressing objects:  62% (82/131)
Receiving objects:  97% (146/150), 16.48 MiB | 10.79 MiB/s/s150)Receiving objects:  19% (29/150)Receiving objects:  29% (44/150)Receiving objects:  30% (45/150)Receiving objects:  51% (77/150), 372.00 KiB | 690.00 KiB/sReceiving objects:  83% (125/150), 372.00 KiB | 690.00 KiB/sReceiving objects:  85% (128/150), 372.00 KiB | 690.00 KiB/sReceiving objects:  86% (129/150), 372.00 KiB | 690.00 KiB/sReceiving objects:  87% (131/150), 372.00 KiB | 690.00 KiB/sReceiving objects:  88% (132/150), 5.22 MiB | 5.08 MiB/s    Receiving objects:  89% (134/150), 5.22 MiB | 5.08 MiB/sReceiving objects:  91% (137/150), 5.22 MiB | 5.08 MiB/sReceiving objects:  93% (140/150), 5.22 MiB | 5.08 MiB/sReceiving objects:  95% (143/150), 16.48

In [2]:
import numpy as np
from rdkit import Chem

EMB_TXT = "./KANO/initial/elementkgontology.embeddings.txt"

def _is_number(x: str) -> bool:
    try:
        float(x); return True
    except Exception:
        return False

def load_w2v_txt(path: str):
    """
    兼容 word2vec txt：
    - 第一行可能是: <num_tokens> <dim>
    - 后续每行: <token> <v1> <v2> ...
    token 可能是 URI/带尖括号，做轻量清洗用于匹配。
    """
    token2vec = {}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        first = f.readline().strip().split()
        has_header = (len(first) == 2 and first[0].isdigit() and first[1].isdigit())
        dim = int(first[1]) if has_header else (len(first) - 1)

        if not has_header:
            # 把第一行当数据行处理
            tok = first[0]
            vec = np.array([float(x) for x in first[1:]], dtype=np.float32)
            token2vec[tok] = vec

        for line in f:
            parts = line.rstrip("\n").split()
            if len(parts) < 2:
                continue
            tok = parts[0]
            vec = np.array([float(x) for x in parts[1:]], dtype=np.float32)
            token2vec[tok] = vec

    # 轻量清洗（用于匹配，不改变原 key）
    cleaned2raw = {}
    for raw in list(token2vec.keys()):
        c = raw.strip()
        if c.startswith("<") and c.endswith(">"):
            c = c[1:-1].strip()
        cleaned2raw[c] = raw
        cleaned2raw[c.lower()] = raw

    # 维度
    any_vec = next(iter(token2vec.values()))
    dim = int(any_vec.shape[0])
    return token2vec, cleaned2raw, dim

token2vec, cleaned2raw, EMB_DIM = load_w2v_txt(EMB_TXT)
print("[OK] Loaded embeddings:", len(token2vec), "dim=", EMB_DIM)


[OK] Loaded embeddings: 2144 dim= 133


In [5]:
import pandas as pd
from collections import Counter

DATA_XLSX = "./Malodors_data.xlsx"   # 你这里上传的文件路径；你本地跑就改成自己的路径

SMILES_COL_CANDIDATES = ["Canonical_SMILES", "Canonical SMILES", "SMILES", "smiles"]

def find_smiles_col(df: pd.DataFrame) -> str:
    for c in SMILES_COL_CANDIDATES:
        if c in df.columns:
            return c
    for c in df.columns:
        if "smiles" in str(c).lower():
            return c
    raise ValueError(f"找不到 SMILES 列。现有列名示例：{df.columns[:30].tolist()} ...")

def canon_smiles(smi: str):
    if smi is None:
        return None
    smi = str(smi).strip()
    if not smi:
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)

def smiles_to_element_counts(csmi: str):
    mol = Chem.MolFromSmiles(csmi)
    if mol is None:
        return None
    cnt = Counter()
    for a in mol.GetAtoms():
        cnt[a.GetSymbol()] += 1
    return cnt

def build_element_token_map(cleaned2raw):
    """
    尝试把元素符号（C, O, Cl, ...）映射到 embedding 里的 token。
    优先级：
      1) 精确命中 symbol
      2) 小写命中 symbol
      3) 命中元素英文名（carbon/chlorine/...）
      4) token 以 /<symbol> 或 #<symbol> 或 :<symbol> 结尾
    """
    pt = Chem.GetPeriodicTable()
    all_tokens_clean = set(cleaned2raw.keys())

    elem2token = {}
    for z in range(1, 119):
        sym = pt.GetElementSymbol(z)  # e.g., "Cl"
        name = pt.GetElementName(z).lower()  # e.g., "chlorine"

        # 1) exact
        if sym in all_tokens_clean:
            elem2token[sym] = cleaned2raw[sym]; continue
        if sym.lower() in all_tokens_clean:
            elem2token[sym] = cleaned2raw[sym.lower()]; continue

        # 2) by name
        if name in all_tokens_clean:
            elem2token[sym] = cleaned2raw[name]; continue

        # 3) suffix heuristics
        hit = None
        suffixes = ["/"+sym, "#"+sym, ":"+sym, "/"+sym.lower(), "#"+sym.lower(), ":"+sym.lower()]
        for t in all_tokens_clean:
            for suf in suffixes:
                if t.endswith(suf):
                    hit = t
                    break
            if hit:
                break
        if hit:
            elem2token[sym] = cleaned2raw[hit]
            continue

        # no hit
        elem2token[sym] = None

    return elem2token

elem2token = build_element_token_map(cleaned2raw)

def elementkg_mol_embedding(elem_counts: Counter, token2vec, elem2token, dim: int):
    # 加权平均：sum(count * e_vec) / sum(count)
    vec = np.zeros((dim,), dtype=np.float32)
    denom = 0.0
    missing = []
    for sym, c in elem_counts.items():
        tok = elem2token.get(sym)
        if tok is None or tok not in token2vec:
            missing.append(sym)
            continue
        vec += float(c) * token2vec[tok]
        denom += float(c)
    if denom > 0:
        vec /= denom
        n = np.linalg.norm(vec) + 1e-12
        vec = vec / n
    return vec, missing

# ===== run =====
df = pd.read_excel(DATA_XLSX)
smiles_col = find_smiles_col(df)
print("[INFO] data shape:", df.shape)
print("[INFO] smiles_col:", smiles_col)

mol_ids, smiles_list, valid_list, miss_list, embs = [], [], [], [], []

for i, smi_raw in enumerate(df[smiles_col].tolist(), start=1):
    mol_id = f"Mol_{i:05d}"
    csmi = canon_smiles(smi_raw)
    mol_ids.append(mol_id)
    smiles_list.append(csmi)

    if csmi is None:
        valid_list.append(False)
        miss_list.append("")
        embs.append(np.zeros((EMB_DIM,), dtype=np.float32))
        continue

    cnt = smiles_to_element_counts(csmi)
    if cnt is None:
        valid_list.append(False)
        miss_list.append("")
        embs.append(np.zeros((EMB_DIM,), dtype=np.float32))
        continue

    vec, missing = elementkg_mol_embedding(cnt, token2vec, elem2token, EMB_DIM)
    valid_list.append(True)
    miss_list.append(",".join(missing))
    embs.append(vec)

embs = np.stack(embs, axis=0)
print("[OK] embeddings:", embs.shape)
print("[INFO] examples of missing element tokens (first 20 rows where missing):")
tmp = [(mol_ids[i], miss_list[i]) for i in range(len(miss_list)) if miss_list[i]]
print(tmp[:20])


[INFO] data shape: (4952, 149)
[INFO] smiles_col: nonStereoSMILES
[OK] embeddings: (4952, 133)
[INFO] examples of missing element tokens (first 20 rows where missing):
[]


[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors
[10:47:41] WARNING: not removing hydrogen atom without neighbors


In [6]:
OUT_DIR = "./Malodors_ElementKG_from_KANO"
os.makedirs(OUT_DIR, exist_ok=True)

out_df = pd.DataFrame({
    "mol_id": mol_ids,
    "smiles": smiles_list,
    "valid": valid_list,
    "missing_elements": miss_list
})
for k in range(EMB_DIM):
    out_df[f"emb_{k:03d}"] = embs[:, k]

out_xlsx = os.path.join(OUT_DIR, "Malodors_ElementKG_embeddings.xlsx")
out_npy  = os.path.join(OUT_DIR, "Malodors_ElementKG_embeddings.npy")

out_df.to_excel(out_xlsx, index=False)
np.save(out_npy, embs)

print("[SAVED]", out_xlsx)
print("[SAVED]", out_npy)


/tmp/ipykernel_38723/3207330111.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[f"emb_{k:03d}"] = embs[:, k]
/tmp/ipykernel_38723/3207330111.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[f"emb_{k:03d}"] = embs[:, k]
/tmp/ipykernel_38723/3207330111.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `n

[SAVED] ./Malodors_ElementKG_from_KANO/Malodors_ElementKG_embeddings.xlsx
[SAVED] ./Malodors_ElementKG_from_KANO/Malodors_ElementKG_embeddings.npy
